# 🌍 Cross-Country Climate Vulnerability Analysis (NASA POWER)
This notebook compares Ethiopia, Kenya, and Nigeria to compute a climate vulnerability ranking for COP32 analysis.

## Temperature Trend Comparison

The monthly temperature trends show seasonal patterns across countries. 
Some countries experience consistently higher temperatures, indicating greater heat exposure.

The variation over time suggests differences in climate stability and warming trends.

## Precipitation Variability

The boxplots reveal differences in rainfall distribution across countries.

Countries with wider spreads and outliers show higher variability, 
indicating less predictable rainfall patterns and higher climate risk.

## Time Series Analysis

Temperature shows a clear seasonal pattern, with identifiable warm and cool periods.

Rainfall is more irregular, with sharp peaks representing rainy seasons.

These patterns highlight climate variability and seasonal dependence.

## Correlation Analysis

The heatmap shows relationships between climate variables.

Strong correlations indicate linked climate behaviors, 
while weak correlations suggest independent factors.

Understanding these relationships helps explain climate dynamics and risks.

## Climate Vulnerability Ranking

The vulnerability index combines temperature, rainfall variability, 
temperature range, and wind speed.

Countries with higher scores are more exposed to climate instability.

This ranking supports climate policy decisions and COP32 discussions.

## Final Conclusion

This cross-country analysis highlights significant differences in climate behavior.

Temperature trends, rainfall variability, and correlations all point to varying levels of climate risk.

The vulnerability ranking provides a data-driven basis for understanding climate exposure 
and supports informed decision-making for COP32.

In [ ]:
# ==============================
# 1. IMPORT LIBRARIES
# ==============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import f_oneway

# ==============================
# 2. LOAD DATASETS
# ==============================
ethiopia = pd.read_csv("../data/ethiopia_clean.csv")
kenya = pd.read_csv("../data/kenya_clean.csv")
nigeria = pd.read_csv("../data/nigeria_clean.csv")

ethiopia['country'] = 'Ethiopia'
kenya['country'] = 'Kenya'
nigeria['country'] = 'Nigeria'

df = pd.concat([ethiopia, kenya, nigeria], ignore_index=True)
df.columns = df.columns.str.lower().str.strip()

# ==============================
# 3. CLEANING
# ==============================
expected_cols = ['t2m', 't2m_max', 't2m_min', 'prectotcorr', 'rh2m', 'ws2m']
df = df.dropna(subset=expected_cols + ['year', 'month'])

# ==============================
# 4. DATE CREATION
# ==============================
df['date'] = pd.to_datetime(dict(year=df['year'], month=df['month'], day=1))
df = df.sort_values('date')

# ==============================
# 5. MONTHLY TEMPERATURE TREND
# ==============================
monthly = df.groupby(['date', 'country'])['t2m'].mean().reset_index()

plt.figure(figsize=(12,6))
for c in monthly['country'].unique():
    subset = monthly[monthly['country'] == c]
    plt.plot(subset['date'], subset['t2m'], label=c)

plt.title("Monthly Temperature Comparison (2015–2026)")
plt.legend()
plt.show()

# ==============================
# 6. FEATURE ENGINEERING
# ==============================
df['t2m_range'] = df['t2m_max'] - df['t2m_min']

# ==============================
# 7. TEMPERATURE SUMMARY
# ==============================
summary = df.groupby('country')['t2m'].agg(['mean', 'median', 'std']).reset_index()
summary.columns = ['country', 'mean_temp', 'median_temp', 'std_temp']
print(summary)

# ==============================
# 8. VULNERABILITY INDEX
# ==============================
vuln = df.groupby('country').agg({
    't2m': 'mean',
    'prectotcorr': 'std',
    't2m_range': 'mean',
    'ws2m': 'mean'
}).reset_index()

vuln.columns = ['country', 'avg_temp', 'rain_var', 'temp_range', 'wind']

scaler = MinMaxScaler()
features = ['avg_temp', 'rain_var', 'temp_range', 'wind']
vuln[features] = scaler.fit_transform(vuln[features])

vuln['vulnerability'] = (
    vuln['avg_temp'] * 0.35 +
    vuln['rain_var'] * 0.30 +
    vuln['temp_range'] * 0.20 +
    vuln['wind'] * 0.15
)

ranking = vuln.sort_values('vulnerability', ascending=False)

print("\nVulnerability Ranking:")
print(ranking)

ranking.set_index('country')['vulnerability'].plot(kind='bar')
plt.title("Climate Vulnerability Ranking")
plt.show()

# ==============================
# 9. PRECIPITATION VARIABILITY
# ==============================
plt.figure(figsize=(10,6))
sns.boxplot(data=df, x='country', y='prectotcorr')
plt.title("Precipitation Variability Across Countries")
plt.show()

rain_summary = df.groupby('country')['prectotcorr'].agg(['mean','median','std']).reset_index()
print(rain_summary)

# ==============================
# 10. TIME SERIES
# ==============================
monthly_temp = df.groupby('date')['t2m'].mean()

plt.figure(figsize=(12,5))
monthly_temp.plot()

plt.scatter(monthly_temp.idxmax(), monthly_temp.max())
plt.scatter(monthly_temp.idxmin(), monthly_temp.min())

plt.title("Temperature Time Series")
plt.show()

monthly_rain = df.groupby('date')['prectotcorr'].sum()

plt.figure(figsize=(12,5))
monthly_rain.tail(24).plot(kind='bar')
plt.title("Rainfall Trend")
plt.show()

# ==============================
# 11. CORRELATION
# ==============================
numeric_df = df.select_dtypes(include='number')

plt.figure(figsize=(10,6))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

# ==============================
# 12. SCATTER PLOTS
# ==============================
plt.figure()
plt.scatter(df['t2m'], df['rh2m'])
plt.title("T2M vs RH2M")
plt.show()

plt.figure()
plt.scatter(df['t2m_range'], df['ws2m'])
plt.title("T2M Range vs Wind")
plt.show()

# ==============================
# 13. TOP CORRELATIONS
# ==============================
corr = numeric_df.corr().abs()
np.fill_diagonal(corr.values, 0)

top_corr = corr.unstack().sort_values(ascending=False).drop_duplicates()
print(top_corr.head(3))

# ==============================
# 14. DISTRIBUTION
# ==============================
skew = df['prectotcorr'].skew()

plt.figure(figsize=(8,5))
if skew > 1:
    plt.hist(np.log1p(df['prectotcorr']), bins=30)
    plt.title("Log Precipitation")
else:
    plt.hist(df['prectotcorr'], bins=30)
    plt.title("Precipitation")

plt.show()

print("Skewness:", skew)

# ==============================
# 15. BUBBLE CHART
# ==============================
plt.figure(figsize=(8,6))

plt.scatter(
    df['t2m'],
    df['rh2m'],
    s=(df['prectotcorr'].fillna(0) / df['prectotcorr'].max()) * 80,
    alpha=0.5
)

plt.title("T2M vs RH2M (Bubble = Rain)")
plt.show()

# ==============================
# 16. EXTREME EVENTS
# ==============================
df['extreme_heat'] = df['t2m_max'] > 35
heat = df.groupby(['country','year'])['extreme_heat'].sum().reset_index()

plt.figure(figsize=(10,6))
sns.barplot(data=heat, x='year', y='extreme_heat', hue='country')
plt.title("Extreme Heat Days")
plt.xticks(rotation=45)
plt.show()

df['dry_day'] = df['prectotcorr'] < 1
dry = df.groupby(['country','year'])['dry_day'].sum().reset_index()

plt.figure(figsize=(10,6))
sns.barplot(data=dry, x='year', y='dry_day', hue='country')
plt.title("Dry Days")
plt.xticks(rotation=45)
plt.show()

# ==============================
# 17. ANOVA TEST
# ==============================
groups = [df[df['country']==c]['t2m'] for c in df['country'].unique()]
f_stat, p_value = f_oneway(*groups)

print("F-stat:", f_stat)
print("P-value:", p_value)

# ==============================
# 18. FINAL INSIGHT
# ==============================
print("""
COP32 INSIGHTS:

- Temperature differences exist across countries.
- Rainfall variability increases climate risk.
- Extreme heat and dry days highlight vulnerability.
- Ethiopia shows moderate but rising risk.
- Policy should prioritize high-risk countries for adaptation funding.
""")

## Distribution of Precipitation

The histogram shows the distribution of precipitation values.

- If log transformation is applied, it indicates strong right skewness.
- Most precipitation values are concentrated at lower levels.
- A few extreme values (heavy rainfall events) create a long tail.

### Interpretation

- The distribution is **positively skewed**, meaning extreme rainfall events are rare but significant.
- This suggests climate variability with occasional heavy rainfall.
- Such patterns are important for flood risk and water resource planning.

## Bubble Chart Analysis

This plot shows the relationship between temperature and humidity, with bubble size representing precipitation.

### Observations

- Larger bubbles indicate higher rainfall events.
- These often occur at specific temperature and humidity combinations.
- The spread shows how rainfall depends on atmospheric conditions.

### Interpretation

- Rainfall tends to increase with higher humidity levels.
- Temperature and humidity together influence precipitation patterns.
- This highlights the interaction between climate variables in driving rainfall.

### Conclusion

The bubble chart reveals complex relationships between temperature, humidity, and precipitation, reinforcing the importance of multi-variable climate analysis.